<a href="https://colab.research.google.com/github/Apur52027/Machine-learing/blob/main/Module_22_XGBoost_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Module 22: XGBoost (Practice Notebook)

### Instructions for Students
- This is a **practice notebook**.
- Complete all **TODO** sections.
- Read the markdown explanations carefully.
- Do not skip evaluation and reflection questions.

Dataset used here is **California Housing (Regression)**.



## 1. Import Required Libraries


In [1]:
# TODO: Import necessary libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor


## 2. Load Dataset (California Housing)


In [2]:
# TODO: Load dataset

from sklearn.datasets import fetch_openml

data = fetch_openml(name="california_housing", version=1, as_frame=True)
X = data.data
y = data.target


## 3. Train-Test Split


In [4]:
# TODO: Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
    )


## 4. Baseline XGBoost Regressor


In [6]:
from re import sub
# TODO: Train baseline model
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    objective='reg:squarederror',
    enable_categorical=True # Added to handle categorical features
)
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='logloss', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)


## 5. Evaluate Baseline Model


In [11]:
# TODO: Evaluate baseline
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

Mean Squared Error: 3289409280.0
R-squared: 0.7489783763885498



## 6. Hyperparameter Tuning with GridSearchCV


In [14]:
# TODO: Define parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 4],
     'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}


### Base Model for Grid Search


In [15]:
# TODO: Base model
xgb_base = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    enable_categorical=True # Added to handle categorical features
)


### Run GridSearchCV


In [17]:
# TODO: Run GridSearchCV
grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1,
    n_jobs=-1
)
grid.fit(X_train, y_train)

Fitting 5 folds for each of 32 candidates, totalling 160 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:29:07] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True,
                                    eval_metric='logloss', feature_types=None,
                                    feature_weights=None, gamma=None,
                                    grow_policy=None, importance_type=None,
                                    interaction_constraints=...
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.1], 'max_depth': [3, 4],
                         'n_estimators': [100, 200], 'subsample': [0.8, 1.0]},
             scoring='neg_mean_squared_error', verbose=1)

In [19]:
print('Best Parameters:', grid.best_params_)
print('Best Score:', grid.best_score_)

Best Parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200, 'subsample': 1.0}
Best Score: -2399751475.2



## 7. Evaluate Tuned Model


In [20]:
# TODO: Evaluate tuned model
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

Mean Squared Error: 2448350976.0
R-squared: 0.813161313533783



## 8. Reflection Questions

1. Did GridSearch improve performance?
2. Which parameter had the biggest effect?
3. What happens if learning_rate is too high?
4. Would you deploy this model? Why?


- Did GridSearch improve performance?

 Yes, GridSearch did improve the model's performance. The baseline model had an R-squared of approximately 0.749, while the tuned model achieved an R-squared of approximately 0.813. This increase in R-squared indicates that the model is now explaining a larger proportion of the variance in the target variable, making it a better predictor.

- Which parameter had the biggest effect?
Pinpointing a single parameter with the 'biggest effect' can be complex without more in-depth analysis (like feature importance for hyperparameters). However, observing the changes from the baseline to the best parameters:

n_estimators increased from 100 to 200.
max_depth increased from 3 to 4.
subsample increased from 0.8 to 1.0.
colsample_bytree increased from 0.8 to 1.0.
learning_rate remained at 0.1.
Generally, n_estimators (number of boosting rounds) and max_depth (the complexity of individual trees) are often very influential in XGBoost. The increases in subsample and colsample_bytree to 1.0 suggest that using more of the data and features during tree building was beneficial, possibly indicating that the baseline was slightly underfitting. It's likely that a combination of these adjustments, especially to n_estimators and max_depth, contributed most significantly to the performance gain.

- What happens if learning_rate is too high?

If the learning_rate is set too high in an XGBoost model, the model can 'overshoot' the optimal solution during its training process. This means that each subsequent tree tries to correct the errors of the preceding ones too aggressively. This aggressive correction can lead to unstable training, preventing the model from converging effectively, or causing it to converge to a suboptimal solution. The result is often higher training and validation errors, and a model that performs poorly on new, unseen data (i.e., it doesn't generalize well).

- Would you deploy this model? Why?

 While the tuned model's R-squared of 0.813 is quite good, the Mean Squared Error (MSE) is still very large (2,448,350,976.0). This translates to a Root Mean Squared Error (RMSE) of approximately $49,481 ($\sqrt{2,448,350,976}$). For predicting median house values, an average error of nearly $50,000 might be considered too high for practical deployment, depending heavily on the specific business context and the typical range of house prices in the dataset.

Therefore, I would not recommend deploying this model immediately without further steps. To improve it, I would consider:

More extensive hyperparameter tuning: Exploring a broader range of parameter values, perhaps with RandomizedSearchCV or more advanced Bayesian optimization techniques.
Feature Engineering: Creating new features or transforming existing ones to provide more predictive power to the model.
Residual Analysis: Examining the model's prediction errors to understand where it performs poorly and to identify any systematic biases.
Robust Cross-validation: Ensuring a reliable cross-validation strategy is in place to confirm the model's generalization ability.
Business Context: Gaining a clearer understanding of the acceptable error margin for the specific application.
Model Interpretability: If necessary, analyzing feature importance to understand the key drivers of the predictions.